# Análisis de los comentarios
---
## Importación de librerías

In [ ]:
# Módulo
import pandas as pd
from pathlib import Path
import re
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import nltk
import numpy as np
import plotly.graph_objects as go
from nltk.corpus import stopwords
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from src.utils.config import load_env_file

load_env_file()

# Descarga usando nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
# Para que se puedan utilizar funciones desde el notebook
import sys
from os import path
root_path = path.abspath(path.join('..', '..'))
if root_path not in sys.path:
    sys.path.append(root_path)
from src.utils.files import read_file
from src.utils.config import reviews

## Carga de datos
Configuración del Minio, poner en True para usar la información de Minio

In [ ]:
use_minio = True # Solo cambiar este parámetro
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df = read_file(reviews, minio)
df.head(2)

## Funciones

In [ ]:
stop_words = set(stopwords.words("english"))
def clean_text(text, stop_words):
    """Se queda solo lo que es texto, quitando stopwords (determinantes, artículos...)"""
    text = text.split()
    text = " ".join(word for word in text if word not in stop_words)
    text = re.sub(r"[^a-z\s]", "", text)
    text = text.split()
    text = " ".join(text)
    return text

In [ ]:
def graph_wordcloud(text,stop_words,is_positive):
    """grafica una nube de palabras dado un string, utiliza un esquema de colores distinto dependiento del tipo de reseña"""
    colormaps = "summer" if is_positive else "autumn"
    wordcloud = WordCloud(width=800, height=400, max_words=100, collocations=True, stopwords=stop_words, background_color="white", min_font_size=10, colormap=colormaps, random_state=42).generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")  
    plt.show()

---
## Análisis
### Separación de comentarios positivos y negativos

In [ ]:
df_positive = df[df["is_positive"]]
df_negative = df[~df["is_positive"]]

In [ ]:
pos_text = " ".join(df_positive["text"].astype(str).tolist())
pos_text = clean_text(pos_text, stop_words)

### Palabras en reviews positivas

In [ ]:
graph_wordcloud(pos_text,stop_words,True)

Hay palabras muy comunes que hacen sombra a aquellas que de verdad tienen carga de significado

In [ ]:
pos_stopwords = stop_words.copy()
# eliminar manualmente palabras filler e irrelevantes
pos_stopwords.update(["even", "one", "way", "make", "play", "get", "see", "found", "thing", "youre", "item", "without", "also"]) 
graph_wordcloud(pos_text,pos_stopwords,True)

In [ ]:
neg_text = " ".join(df_negative["text"].astype(str).tolist())
neg_text = clean_text(neg_text, stop_words)

### Palabras en reviews negativas

In [ ]:
graph_wordcloud(neg_text,stop_words,False)

In [ ]:
neg_stopwords = stop_words.copy()
# hay que quitar palabras trampa que probablemente sean "not good, not fun", etc.
neg_stopwords.update(["game", "one", "thing", "really","feel", "even", "good", "fun", "going", "way", "also", "play", "got", "put", "know", "thats","like", "get", "games", "make","heartsheartsheartshearts","much", "made", "want"])
graph_wordcloud(neg_text,neg_stopwords,False)

In [ ]:
pos_words = pos_text.split()
freq = Counter(pos_words)
pos_dict = {}
for word, counter in freq.most_common():
    pos_dict[word] = counter
pos_dict

In [ ]:
neg_words = neg_text.split()
freq2 = Counter(neg_words)
neg_dict = {}
for word, counter in freq2.most_common():
    neg_dict[word] = counter
neg_dict

### Análisis con TF-IDF

In [ ]:
pos_text = "\n".join(df_positive["text"].astype(str).tolist())
neg_text = "\n".join(df_negative["text"].astype(str).tolist())
pos_reviews = [clean_text(r,stop_words) for r in pos_text.split("\n") if r.strip()] # documentos de reseñas positivas [rev1, rev2, rev3...]
neg_reviews = [clean_text(r,stop_words) for r in neg_text.split("\n") if r.strip()] # documentos de reseñas negativas

documents = pos_reviews + neg_reviews # lo que se denomina corpus

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    max_df=0.75, # ignorar palabras que aparecen en más del 85% de 
    min_df=5, 
    max_features=5000 
)

X = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

labels = np.array([1]*len(pos_reviews) + [0]*len(neg_reviews))
# se hace la media, pues hay un desbalance entre reseñas positivas y negativas
pos_mean = X[labels == 1].mean(axis=0).A1  # pos_mean[i] es la importancia media de palabra i en las reviews positivas 
neg_mean = X[labels == 0].mean(axis=0).A1  # analogo a pos_mean


diff = pos_mean - neg_mean # diferencia de importancias. Aquellas palabras que sean importantes en las reseñas positivas y no
                           # en las negativas serán muy positivas. Muy negativas en el caso contrario.
top_pos = np.argsort(diff)[-100:] # indices que ordenarían el array diff. Al final están los "pesos" las palabras importantes en las reviews positivas (y no en importante en las negativas)
top_neg = np.argsort(diff)[:100]  # al principio están los valores más negativos (pequeños) que corresponde a las palabras más importantes en las reseñas negativas (y no en importante en las positivas)


print("Top palabras positivas:")
print([feature_names[i] for i in reversed(top_pos)])

print("Top palabras negativas:")
print([feature_names[i] for i in top_neg])

### Nube de palabras con TF-IDF

In [ ]:
# diccionario de palabras con sus pesos
top_positive_words = {feature_names[i]: abs(diff[i]) for i in reversed(top_pos)}
top_negative_words = {feature_names[i]: abs(diff[i]) for i in top_neg}

cloud_pos = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="summer",
    collocations = False,
    random_state=42
).generate_from_frequencies(top_positive_words)

plt.figure(figsize=(10,5))
plt.imshow(cloud_pos)
plt.axis("off")
plt.show()


cloud_neg = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="autumn",
    collocations = False,
    random_state=42
).generate_from_frequencies(top_negative_words)


plt.figure(figsize=(10,5))
plt.imshow(cloud_neg)
plt.axis("off")
plt.show()

### Longitud de la reseñas


In [ ]:
def split_text(text):
    return text.split()

df_positive["len_text"] = df_positive["text"].apply(split_text).apply(len)
df_negative["len_text"] = df_negative["text"].apply(split_text).apply(len)
media_pos = df_positive["len_text"].mean()
media_neg = df_negative["len_text"].mean()
fig_pos = go.Figure()
fig_pos.add_trace(go.Histogram(x = df_positive["len_text"], name= "positivas"))
fig_pos.add_vline(x=media_pos, line_width=3, line_dash="dot", line_color="blue", annotation_text=f"Media: {media_pos.round(2)}", annotation_position = "top right")
fig_pos.update_layout(barmode = "overlay", 
                  title = "Distribución de la longitud del texto para reseñas positivas y negativas", 
                  xaxis_title = "número de palabras",
                  yaxis_title = "Número de reseñas")

fig_pos.show()              
fig_neg = go.Figure()
fig_neg.add_vline(x=media_neg, line_width=3, line_dash="dot", line_color="blue", annotation_text=f"Media: {media_neg.round(2)}", annotation_position = "top right")
fig_neg.add_trace(go.Histogram(x = df_negative["len_text"],name= "negativas"))
fig_neg.update_layout(barmode = "overlay", 
                  title = "Distribución de la longitud del texto para reseñas negativas", 
                  xaxis_title = "Número de palabras",
                  yaxis_title = "Número de reseñas")
fig_neg.show()   

La media no es robusta ante valores atípicos, vamos a ver la moda


In [ ]:
print(f"Moda de número de palabras en reseñas positvas: {df_positive["len_text"].mode().values[0]}")
print(f"Moda de número de palabras en reseñas negativas: {df_negative["len_text"].mode().values[0]}")

Se puede observar que en general, las reseñas negativas son un poco más extensas que las reseñas positivas, probablemente sea porque suele ser una postura minoritaria
y el que escribe la review siente la necesidad de justificar su opinión.